# 02. Geospatial Entity Resolution, Cross-Source OSM Verification & Corporate Deduplication

Resolves Stage-01 DENUE establishments against OpenStreetMap geometries, clusters multi-plant corporate campuses using NLP (TF-IDF + RapidFuzz), enriches decision-maker contact details from corporate web domains, and generates LinkedIn ABM query hooks.

### Pipeline Capabilities
1. **Cross-Source Spatial Verification:** Verifies physical storefronts within 50m of OpenStreetMap POIs (ODbL horizontal separation maintained).
2. **NLP Corporate Campus Deduplication:** Clusters fragmented plant/office records (e.g. *Aumovio Matriz vs Planta*, *Seglo*, *Bodycote*, *Ryder*) into canonical parent entities.
3. **Deep Contact & Decision-Maker Enrichment:** Audits emails into decision-maker roles (RH/Purchasing) and extracts B2B contact forms & LinkedIn URLs.
4. **Interactive Campus Resolution Map:** Visualizes corporate headquarters, connected plant sites, and verified storefronts in the last cell.

In [1]:
# Cell 2.1: Load Stage-01 Enriched Dataset & ODbL OSM Layer
from pathlib import Path
import polars as pl

raw_dir = Path("data/raw")
raw_files = list(raw_dir.glob("*.parquet"))

if raw_files:
    source_file = raw_files[0]
    print(f"Loading cached Stage-01 handoff data from {source_file}")
    df = pl.read_parquet(source_file)
else:
    fixture_file = Path("fixtures/synthetic_denue_sample.parquet")
    print(f"Loading fixture fallback from {fixture_file}")
    df = pl.read_parquet(fixture_file)
    if "distance_km" not in df.columns:
        df = df.with_columns(pl.lit(1.5).alias("distance_km"))

# Load ODbL OpenStreetMap layer if available
osm_file = Path("data/processed/osm_amenities.parquet")
if osm_file.exists():
    osm_df = pl.read_parquet(osm_file)
    print(f"Loaded ODbL OSM layer from {osm_file}: {len(osm_df)} POIs")
else:
    osm_df = pl.DataFrame({
        "osm_id": pl.Series([], dtype=pl.Utf8),
        "lat": pl.Series([], dtype=pl.Float64),
        "lon": pl.Series([], dtype=pl.Float64),
        "name": pl.Series([], dtype=pl.Utf8),
        "amenity": pl.Series([], dtype=pl.Utf8),
        "office": pl.Series([], dtype=pl.Utf8),
        "shop": pl.Series([], dtype=pl.Utf8),
    })
    print("OSM amenities layer not found; proceeding with empty context.")

print(f"Total Stage-01 records loaded: {len(df):,}")

Loading cached Stage-01 handoff data from data/raw/denue_state_11_radius_5km.parquet
Loaded ODbL OSM layer from data/processed/osm_amenities.parquet: 72 POIs
Total Stage-01 records loaded: 398


In [2]:
# MIGRATES TO: src/mx_osint/resolution/spatial_blocker.py::spatial_join_osm_context
import math
import polars as pl


def encode_geohash(latitude: float, longitude: float, precision: int = 6) -> str:
    """Encodes coordinates into standard geohash string for localized O(n*k) blocking."""
    base32 = "0123456789bcdefghjkmnpqrstuvwxyz"
    lat_interval, lon_interval = [-90.0, 90.0], [-180.0, 180.0]
    geohash, bits, bit, ch, even = [], [16, 8, 4, 2, 1], 0, 0, True
    while len(geohash) < precision:
        if even:
            mid = (lon_interval[0] + lon_interval[1]) / 2.0
            if longitude > mid:
                ch |= bits[bit]
                lon_interval[0] = mid
            else:
                lon_interval[1] = mid
        else:
            mid = (lat_interval[0] + lat_interval[1]) / 2.0
            if latitude > mid:
                ch |= bits[bit]
                lat_interval[0] = mid
            else:
                lat_interval[1] = mid
        even = not even
        if bit < 4:
            bit += 1
        else:
            geohash.append(base32[ch])
            bit, ch = 0, 0
    return "".join(geohash)


def assign_geohash_block(df: pl.DataFrame, precision: int = 6) -> pl.DataFrame:
    """Assigns geohash spatial block column."""
    hashes = [encode_geohash(lat, lon, precision) for lat, lon in zip(df["lat"].to_list(), df["lon"].to_list())]
    return df.with_columns(pl.Series("geohash", hashes))


def spatial_join_osm_context(
    denue_df: pl.DataFrame,
    osm_df: pl.DataFrame,
    max_dist_m: float = 50.0,
) -> pl.DataFrame:
    """
    Cross-source spatial resolution between DENUE and OSM amenities respecting ODbL boundaries.
    Matches within 50m verify physical storefront/facility existence.
    """
    if len(osm_df) == 0:
        return denue_df.with_columns([
            pl.lit(None).cast(pl.Utf8).alias("nearest_osm_id"),
            pl.lit(None).cast(pl.Float64).alias("osm_match_dist_m"),
            pl.lit(False).alias("osm_verified_storefront"),
            pl.lit("").alias("osm_poi_context"),
        ])

    osm_records = osm_df.select(["osm_id", "lat", "lon", "name", "amenity", "office", "shop"]).to_dicts()
    r_earth_m = 6371000.0
    rad = math.pi / 180.0

    nearest_ids = []
    nearest_dists = []
    verified_flags = []
    contexts = []

    for row in denue_df.iter_rows(named=True):
        d_lat, d_lon = row["lat"], row["lon"]
        best_dist = float("inf")
        best_osm = None

        for osm in osm_records:
            o_lat, o_lon = osm["lat"], osm["lon"]
            dlat = (o_lat - d_lat) * rad
            dlon = (o_lon - d_lon) * rad
            a = math.sin(dlat / 2.0) ** 2 + math.cos(d_lat * rad) * math.cos(o_lat * rad) * math.sin(dlon / 2.0) ** 2
            dist_m = 2.0 * math.asin(math.sqrt(a)) * r_earth_m

            if dist_m < best_dist:
                best_dist = dist_m
                best_osm = osm

        if best_osm and best_dist <= 250.0:
            nearest_ids.append(str(best_osm["osm_id"]))
            nearest_dists.append(round(best_dist, 1))
            verified_flags.append(best_dist <= max_dist_m)
            ctx = best_osm.get("name") or best_osm.get("amenity") or best_osm.get("office") or best_osm.get("shop") or ""
            contexts.append(str(ctx))
        else:
            nearest_ids.append(None)
            nearest_dists.append(None)
            verified_flags.append(False)
            contexts.append("")

    return denue_df.with_columns([
        pl.Series("nearest_osm_id", nearest_ids, dtype=pl.Utf8),
        pl.Series("osm_match_dist_m", nearest_dists, dtype=pl.Float64),
        pl.Series("osm_verified_storefront", verified_flags, dtype=pl.Boolean),
        pl.Series("osm_poi_context", contexts, dtype=pl.Utf8),
    ])

In [3]:
# MIGRATES TO: src/mx_osint/resolution/mex_name_normalizer.py::normalize_mexican_legal_name
import hashlib
import re
import unicodedata
import polars as pl
from rapidfuzz import fuzz

LEGAL_SUFFIX_REGEX = re.compile(
    r"\b("
    r"S\.?\s*A\.?\s*P\.?\s*I\.?\s*(?:DE\s*C\.?\s*V\.?)?|"
    r"S\.?\s*A\.?\s*B\.?\s*(?:DE\s*C\.?\s*V\.?)?|"
    r"S\.?\s*A\.?\s*DE\s*C\.?\s*V\.?|"
    r"S\.?\s*DE\s*R\.?\s*L\.?\s*(?:DE\s*C\.?\s*V\.?)?|"
    r"S\.?\s*C\.?\s*DE\s*R\.?\s*L\.?|"
    r"S\.?\s*C\.?|"
    r"A\.?\s*C\.?|"
    r"S\.?\s*A\.?|"
    r"SAB\s*DE\s*CV"
    r")\b",
    re.IGNORECASE
)

BRANCH_QUALIFIER_REGEX = re.compile(
    r"\b(PLANTA|MATRIZ|SUCURSAL|CORPORATIVO|CAMPUS|SILAO|LEON|GUANAJUATO|BAJIO|MEXICO)\b",
    re.IGNORECASE
)

PUNCTUATION_REGEX = re.compile(r"[,.\-'\"]")


def normalize_mexican_legal_name(name: str) -> str:
    """Normalizes trade/legal names by stripping accents, Mexican corporate suffixes, and punctuation."""
    if not name or not isinstance(name, str):
        return ""
    text = unicodedata.normalize("NFKD", name)
    text = "".join(c for c in text if not unicodedata.combining(c)).upper()
    text = LEGAL_SUFFIX_REGEX.sub(" ", text)
    text = PUNCTUATION_REGEX.sub(" ", text)
    return " ".join(text.split()).strip()


def strip_corporate_branch_qualifiers(name: str) -> str:
    """Strips secondary branch/plant descriptors (e.g. PLANTA, MATRIZ, SILAO) for campus parent resolution."""
    cleaned = normalize_mexican_legal_name(name)
    cleaned = BRANCH_QUALIFIER_REGEX.sub(" ", cleaned)
    cleaned = PUNCTUATION_REGEX.sub(" ", cleaned)
    return " ".join(cleaned.split()).strip()


# Verify all 10 standard test cases pass
test_cases = [
    ("GRUPO INDUSTRIAL DEL BAJÍO S.A. DE C.V.", "GRUPO INDUSTRIAL DEL BAJIO"),
    ("TECNOLOGIA Y SOFTWARE S.A.P.I. DE C.V.", "TECNOLOGIA Y SOFTWARE"),
    ("LOGISTICA NACIONAL S. DE R.L. DE C.V.", "LOGISTICA NACIONAL"),
    ("SERVICIOS CONTABLES Y FISCALES S.C.", "SERVICIOS CONTABLES Y FISCALES"),
    ("FUNDACION COMUNITARIA A.C.", "FUNDACION COMUNITARIA"),
    ("COMERCIALIZADORA AGRICOLA S.A.", "COMERCIALIZADORA AGRICOLA"),
    ("COOPERATIVA DE TRANSPORTE S.C. DE R.L.", "COOPERATIVA DE TRANSPORTE"),
    ("BANCO MERCANTIL SAB DE CV", "BANCO MERCANTIL"),
    ("CORPORATIVO NACIONAL, S.A. B. DE C.V.", "CORPORATIVO NACIONAL"),
    ("FARMACIAS DE MÉXICO, S.A. DE C.V.", "FARMACIAS DE MEXICO"),
]
for raw, expected in test_cases:
    assert normalize_mexican_legal_name(raw) == expected, f"Mismatch on {raw}"
print("✅ All 10 legal normalizer unit test assertions passed.")


def cluster_corporate_entities(
    df: pl.DataFrame,
    fuzzy_threshold: float = 0.85,
) -> pl.DataFrame:
    """
    Clusters multi-plant corporate campuses and duplicate branch entries using
    RapidFuzz WRatio and branch-qualifier stripping across legal and commercial names.
    """
    name_col = "nombre_estab" if "nombre_estab" in df.columns else "business_name"
    legal_col = "raz_social" if "raz_social" in df.columns else "legal_name"
    id_col = "id" if "id" in df.columns else "row_id"

    norm_commercial = [normalize_mexican_legal_name(str(n)) for n in df[name_col].to_list()]
    parent_commercial = [strip_corporate_branch_qualifiers(str(n)) for n in df[name_col].to_list()]
    norm_legal = [normalize_mexican_legal_name(str(l)) if l else "" for l in df[legal_col].to_list()]
    parent_legal = [strip_corporate_branch_qualifiers(str(l)) if l else "" for l in df[legal_col].to_list()]

    df_prep = df.with_columns([
        pl.Series("norm_name", norm_commercial),
        pl.Series("parent_name", parent_commercial),
        pl.Series("norm_legal", norm_legal),
        pl.Series("parent_legal", parent_legal),
    ])

    canonical_parent_keys = []
    for p_leg, p_comm, gh in zip(parent_legal, parent_commercial, df_prep["geohash"].to_list()):
        key_name = p_leg if (p_leg and len(p_leg) >= 4) else p_comm
        canonical_parent_keys.append(f"{key_name}_{gh[:3]}")

    df_prep = df_prep.with_columns(pl.Series("parent_cluster_key", canonical_parent_keys))

    is_duplicate = [False] * len(df_prep)
    campus_names = []
    canonical_ids = []
    records = df_prep.to_dicts()

    from collections import defaultdict
    groups = defaultdict(list)
    for idx, r in enumerate(records):
        groups[r["parent_cluster_key"]].append((idx, r))

    for key, items in groups.items():
        def sort_priority(item):
            r = item[1]
            score = r.get("b2b_5d_score", 0.0) or 0.0
            is_matriz = 1 if "MATRIZ" in str(r.get(name_col, "")).upper() else 0
            return (is_matriz, score)

        sorted_items = sorted(items, key=sort_priority, reverse=True)
        primary_idx, primary_record = sorted_items[0]
        canonical_id = f"ent_{hashlib.sha1(key.encode('utf-8')).hexdigest()[:12]}"
        primary_name = primary_record[name_col]

        for idx, r in sorted_items:
            canonical_ids.append((idx, canonical_id))
            campus_names.append((idx, primary_name))
            if idx != primary_idx and len(sorted_items) > 1:
                name_score = fuzz.WRatio(primary_record["norm_name"], r["norm_name"]) / 100.0
                legal_score = fuzz.WRatio(primary_record["norm_legal"], r["norm_legal"]) / 100.0 if primary_record["norm_legal"] and r["norm_legal"] else 0.0
                if max(name_score, legal_score) >= fuzzy_threshold or (primary_record["norm_legal"] and primary_record["norm_legal"] == r["norm_legal"]):
                    is_duplicate[idx] = True

    canonical_ids_sorted = [cid for idx, cid in sorted(canonical_ids, key=lambda x: x[0])]
    campus_names_sorted = [cname for idx, cname in sorted(campus_names, key=lambda x: x[0])]

    df_resolved = df_prep.with_columns([
        pl.Series("canonical_entity_id", canonical_ids_sorted),
        pl.Series("canonical_parent_name", campus_names_sorted),
        pl.Series("is_duplicate", is_duplicate),
        pl.Series("entity_id", [f"rec_{hashlib.sha1(str(i).encode()).hexdigest()[:10]}" for i in df_prep[id_col].to_list()]),
    ])

    campus_payroll_sum = (
        df_resolved.group_by("canonical_entity_id")
        .agg([
            pl.len().alias("campus_site_count"),
            (pl.col("est_monthly_payroll_mxn").sum() if "est_monthly_payroll_mxn" in df_resolved.columns else pl.lit(0.0)).alias("campus_consolidated_payroll_mxn"),
        ])
    )

    return df_resolved.join(campus_payroll_sum, on="canonical_entity_id", how="left")

✅ All 10 legal normalizer unit test assertions passed.


In [4]:
# MIGRATES TO: src/mx_osint/resolution/contact_enricher.py::enrich_b2b_contacts
from concurrent.futures import ThreadPoolExecutor, as_completed
import re
from urllib.parse import quote, urljoin
import polars as pl
import requests
from loguru import logger

EMAIL_REGEX = re.compile(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}")
LINKEDIN_COMPANY_REGEX = re.compile(r"https?://(?:www\.)?linkedin\.com/company/[a-zA-Z0-9_-]+", re.IGNORECASE)


def classify_b2b_email_role(email: str | None) -> str:
    """Classifies email into decision maker, functional generic inbox, corporate direct, or free personal."""
    if not email or not isinstance(email, str) or "@" not in email:
        return "none"
    clean_em = email.strip().lower()
    free_domains = ["gmail.com", "hotmail.com", "outlook.com", "yahoo.com", "live.com", "prodigy.net.mx"]
    if any(clean_em.endswith(f"@{d}") for d in free_domains):
        return "free_personal"

    user_part = clean_em.split("@")[0]
    decision_makers = ["director", "gerente", "rh", "recursoshumanos", "purchasing", "compras", "talento", "ceo", "cfo", "planta"]
    generic_roles = ["contacto", "info", "recepcion", "atencion", "ventas", "servicio", "facturacion", "admin"]

    if any(dm in user_part for dm in decision_makers):
        return "decision_maker_role"
    if any(gen in user_part for gen in generic_roles):
        return "generic_role_inbox"
    return "corporate_direct"


def scrape_website_contacts(url: str, timeout: float = 3.5) -> dict:
    """Scrapes official company website for direct decision-maker emails and LinkedIn company URLs."""
    if not url or not isinstance(url, str):
        return {"scraped_email": None, "scraped_linkedin": None}

    norm_url = url.strip()
    if not norm_url.startswith("http"):
        norm_url = f"https://{norm_url}"

    headers = {"User-Agent": "MexicoCommercialOSINT/1.0 (B2B Lead Verifier; contact@paseoms.mx)"}
    target_paths = ["", "/contacto", "/contact", "/nosotros", "/directorio"]
    found_emails, found_linkedins = set(), set()

    for path in target_paths:
        sub_url = urljoin(norm_url, path)
        try:
            resp = requests.get(sub_url, headers=headers, timeout=timeout, allow_redirects=True)
            if resp.status_code == 200:
                text = resp.text
                for em in EMAIL_REGEX.findall(text):
                    if not any(em.lower().endswith(ext) for ext in [".png", ".jpg", ".jpeg", ".gif", ".svg", ".webp"]):
                        found_emails.add(em.strip().lower())
                for li in LINKEDIN_COMPANY_REGEX.findall(text):
                    found_linkedins.add(li.strip())
                if found_emails and found_linkedins:
                    break
        except Exception:
            continue

    chosen_email = None
    for em in sorted(found_emails):
        role = classify_b2b_email_role(em)
        if role == "decision_maker_role":
            chosen_email = em
            break
    if not chosen_email and found_emails:
        chosen_email = sorted(found_emails)[0]

    chosen_linkedin = sorted(found_linkedins)[0] if found_linkedins else None
    return {"scraped_email": chosen_email, "scraped_linkedin": chosen_linkedin}


def enrich_b2b_contacts(
    df: pl.DataFrame,
    max_workers: int = 8,
) -> pl.DataFrame:
    """
    Enriches actionable B2B targets with email role classification, website scraping,
    and auto-generated LinkedIn ABM search query hooks.
    """
    existing_roles = [classify_b2b_email_role(em) for em in df["email"].to_list() if "email" in df.columns]
    if not existing_roles and "correoelec" in df.columns:
        existing_roles = [classify_b2b_email_role(em) for em in df["correoelec"].to_list()]

    linkedin_searches = []
    for name, loc in zip(df["canonical_parent_name"].to_list(), df["municipality"].to_list() if "municipality" in df.columns else ["Silao"] * len(df)):
        clean_query = quote(f"{name} {loc}")
        linkedin_searches.append(f"https://www.linkedin.com/search/results/people/?keywords={clean_query}")

    web_col = "website" if "website" in df.columns else "www"
    records = df.to_dicts()
    scraped_emails = [None] * len(records)
    scraped_linkedins = [None] * len(records)

    targets_to_scrape = [
        (i, r[web_col]) for i, r in enumerate(records)
        if r.get("b2b_status") == "target" and r.get(web_col) and r.get(web_col) != ""
    ][:15]

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_idx = {executor.submit(scrape_website_contacts, url): idx for idx, url in targets_to_scrape}
        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            try:
                res = future.result()
                scraped_emails[idx] = res.get("scraped_email")
                scraped_linkedins[idx] = res.get("scraped_linkedin")
            except Exception:
                pass

    final_emails = []
    raw_emails = df["email"].to_list() if "email" in df.columns else df["correoelec"].to_list()
    for raw, sc in zip(raw_emails, scraped_emails):
        final_emails.append(sc if (sc and not raw) else raw)

    return df.with_columns([
        pl.Series("email_role_tier", existing_roles),
        pl.Series("linkedin_search_url", linkedin_searches),
        pl.Series("scraped_company_linkedin", scraped_linkedins),
        pl.Series("verified_contact_email", final_emails),
    ])

In [5]:
# MIGRATES TO: src/mx_osint/resolution/entity_resolver.py::persist_resolved
from pathlib import Path
import shutil
import folium
from IPython.display import display
from loguru import logger
import polars as pl


def persist_resolved(
    df: pl.DataFrame,
    output_path: Path = Path("data/processed/entities_resolved.parquet"),
) -> Path:
    """Persists deduplicated, campus-clustered, and contact-enriched entity table for Stage 03."""
    output_path.parent.mkdir(parents=True, exist_ok=True)

    handoff_df = df.with_columns([
        pl.col("business_name").alias("nombre_estab") if "nombre_estab" not in df.columns else pl.col("nombre_estab"),
        pl.col("legal_name").alias("raz_social") if "raz_social" not in df.columns else pl.col("raz_social"),
        pl.col("scian_code").alias("codigo_act") if "codigo_act" not in df.columns else pl.col("codigo_act"),
        pl.col("scian_description").alias("nombre_act") if "nombre_act" not in df.columns else pl.col("nombre_act"),
        pl.col("employee_strata").alias("per_ocu") if "per_ocu" not in df.columns else pl.col("per_ocu"),
        pl.col("phone").alias("telefono") if "telefono" not in df.columns else pl.col("telefono"),
        pl.col("verified_contact_email").alias("correoelec") if "correoelec" not in df.columns else pl.col("correoelec"),
        pl.col("website").alias("www") if "www" not in df.columns else pl.col("www"),
        pl.col("municipality").alias("mun") if "mun" not in df.columns else pl.col("mun"),
    ])

    handoff_df.write_parquet(output_path)

    for peer_root in (Path(".."), Path("notebooks")):
        if (peer_root / "data" / "processed").exists():
            peer_dest = peer_root / "data" / "processed" / output_path.name
            if peer_dest.resolve() != output_path.resolve():
                shutil.copy2(output_path, peer_dest)

    total = len(handoff_df)
    dups = handoff_df.filter(pl.col("is_duplicate") == True).height
    campuses = handoff_df["canonical_entity_id"].n_unique()
    osm_verified = handoff_df.filter(pl.col("osm_verified_storefront") == True).height

    print("\n" + "=" * 78)
    print(f"🏛️ RESUMEN EJECUTIVO — RESOLUCIÓN DE ENTIDADES & CLUSTERING CORPORATIVO")
    print("=" * 78)
    print(f"Total registros procesados:              {total:,}")
    print(f"Sucursales / Plantas duplicadas flagged:  {dups:,} ({dups/max(total, 1)*100:.1f}%)")
    print(f"Entidades / Campus canónicos únicos:     {campuses:,}")
    print(f"Fachadas verificadas con OSM (<= 50m):   {osm_verified:,} ({osm_verified/max(total, 1)*100:.1f}%)")
    print(f"Guardado exitosamente en:                {output_path}")
    print("=" * 78)
    return output_path


def create_resolution_map(
    df: pl.DataFrame,
    target_lat: float = 20.953791996685183,
    target_lon: float = -101.41605322904304,
    output_path: Path = Path("data/processed/entities_resolved_map.html"),
) -> folium.Map:
    """
    Renders interactive Folium map showing resolved corporate entities, clustered multi-plant
    campuses, OSM storefront verification flags, and LinkedIn ABM query links.
    """
    m = folium.Map(location=[target_lat, target_lon], zoom_start=14, tiles=None)

    folium.TileLayer(
        tiles="https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",
        attr="&copy; OpenStreetMap contributors &copy; CARTO",
        name="🗺️ Mapa Claro (CARTO Positron)",
        overlay=False,
        control=True,
    ).add_to(m)

    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
        attr="Tiles &copy; Esri",
        name="🏙️ Esri World Street Map",
        overlay=False,
        control=True,
    ).add_to(m)

    for r_m, col in [(1000, "#d62728"), (3000, "#ff7f0e")]:
        folium.Circle(location=[target_lat, target_lon], radius=r_m, color=col, weight=1.5, fill=False, dash_array="5, 5").add_to(m)

    fg_primary = folium.FeatureGroup(name="🏢 Campus Corporativo Canónico (Headquarters)", show=True)
    fg_branches = folium.FeatureGroup(name="🏭 Plantas / Sucursales Secundarias Enlazadas", show=True)
    fg_osm_verified = folium.FeatureGroup(name="✅ Fachada Verificada con OSM (<= 50m)", show=True)

    campus_groups = {}
    for r in df.iter_rows(named=True):
        cid = r["canonical_entity_id"]
        if cid not in campus_groups:
            campus_groups[cid] = []
        campus_groups[cid].append(r)

    for cid, site_list in campus_groups.items():
        if len(site_list) > 1:
            primary = [s for s in site_list if not s.get("is_duplicate")][0] if any(not s.get("is_duplicate") for s in site_list) else site_list[0]
            for branch in site_list:
                if branch["row_id"] != primary["row_id"]:
                    folium.PolyLine(
                        locations=[[primary["lat"], primary["lon"]], [branch["lat"], branch["lon"]]],
                        color="#7b3294",
                        weight=2.5,
                        dash_array="4, 6",
                        opacity=0.7,
                        tooltip=f"Enlace de Campus Corporativo: {primary['canonical_parent_name']}",
                    ).add_to(fg_branches)

    for r in df.iter_rows(named=True):
        is_dup = r.get("is_duplicate", False)
        osm_v = r.get("osm_verified_storefront", False)
        color = "#7b3294" if is_dup else "#1b7837"
        radius = 5.5 if is_dup else 8.5
        badge = "🏭 SUCURSAL / PLANTA SECUNDARIA" if is_dup else "🏢 SEDE CANÓNICA DE DECISIÓN"

        popup_html = f"""
        <div style="min-width: 270px; font-family: Arial, sans-serif; font-size: 12px;">
            <b style="color:{color}; font-size:14px;">{r['business_name']}</b><br>
            <span style="background-color:{color}; color:white; padding:2px 5px; border-radius:3px; font-size:10px;">{badge}</span><br>
            <hr style="margin: 5px 0;">
            <b>🏛️ Grupo Corporativo:</b> {r['canonical_parent_name']} ({r['campus_site_count']} sitios en Silao)<br>
            <b>👥 Nómina Consolidada Grupo:</b> ${r.get('campus_consolidated_payroll_mxn', 0)/1e6:.2f}M MXN/mes<br>
            <b>📍 OSM Fachada Verificada:</b> {'✅ SÍ (a ' + str(r.get('osm_match_dist_m')) + 'm)' if osm_v else 'Sin match <= 50m'}<br>
            <b>📧 Email Verificado:</b> {r.get('verified_contact_email') or 'N/A'} ({r.get('email_role_tier', 'N/A')})<br>
            <hr style="margin: 5px 0;">
            <a href="{r.get('linkedin_search_url', '#')}" target="_blank" style="color:#0077b5; font-weight:bold;">🔍 Buscar Decisores en LinkedIn Sales Navigator &rarr;</a>
        </div>
        """

        mkr = folium.CircleMarker(
            location=[r["lat"], r["lon"]],
            radius=radius,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.85,
            weight=1.5,
            popup=folium.Popup(popup_html, max_width=350),
            tooltip=f"[{r['canonical_parent_name']}] {r['business_name']}",
        )
        if is_dup:
            mkr.add_to(fg_branches)
        else:
            mkr.add_to(fg_primary)

        if osm_v:
            folium.CircleMarker(
                location=[r["lat"], r["lon"]],
                radius=12.0,
                color="#00aa00",
                weight=1,
                fill=False,
                dash_array="3, 3",
                tooltip=f"✅ Storefront OSM confirmado: {r.get('osm_poi_context', '')}",
            ).add_to(fg_osm_verified)

    fg_primary.add_to(m)
    fg_branches.add_to(m)
    fg_osm_verified.add_to(m)

    folium.Marker(
        location=[target_lat, target_lon],
        popup="<b>☕ The Coffee — Paseo MS</b>",
        tooltip="🎯 The Coffee — Paseo MS",
        icon=folium.Icon(color="red", icon="coffee", prefix="fa"),
    ).add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    m.save(str(output_path))
    logger.info(f"✅ Interactive Resolution Map saved -> {output_path}")
    return m


# ═══════════════════════════════════════════════════════════════════════════
# EXECUTE COMPLETE STAGE-02 PIPELINE & EMBED INTERACTIVE RESOLUTION MAP
# ═══════════════════════════════════════════════════════════════════════════
df_blocked = assign_geohash_block(df, precision=6)
df_osm_joined = spatial_join_osm_context(df_blocked, osm_df, max_dist_m=50.0)
df_clustered = cluster_corporate_entities(df_osm_joined, fuzzy_threshold=0.85)
df_final_resolved = enrich_b2b_contacts(df_clustered, max_workers=6)

out_parquet = persist_resolved(df_final_resolved)

resolution_map = create_resolution_map(df_final_resolved)
display(resolution_map)


🏛️ RESUMEN EJECUTIVO — RESOLUCIÓN DE ENTIDADES & CLUSTERING CORPORATIVO
Total registros procesados:              398
Sucursales / Plantas duplicadas flagged:  68 (17.1%)
Entidades / Campus canónicos únicos:     329
Fachadas verificadas con OSM (<= 50m):   65 (16.3%)
Guardado exitosamente en:                data/processed/entities_resolved.parquet


2026-10-24 21:46:58.679 | INFO     | __main__:create_resolution_map:172 - ✅ Interactive Resolution Map saved -> data/processed/entities_resolved_map.html
